In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re

In [3]:
with open('../output/mat/all-questions.md', 'r') as qf:
    questions = qf.readlines()
len(questions)

60

In [58]:
with open('output/inv-trig/rep-questions-approaches.md', 'r') as af:
    approaches = af.readlines()

In [59]:
len(approaches)

15

In [5]:
model = SentenceTransformer("BAAI/bge-m3")

In [6]:
embeddings = model.encode(questions)

/pytorch/third_party/ideep/mkl-dnn/src/cpu/aarch64/xbyak_aarch64/src/util_impl_linux.h, 451: Can't read MIDR_EL1 sysfs entry


In [7]:
# Cluster using KMeans
n_clusters = 18
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

In [8]:
# Find central question in each cluster
representatives = []
for cluster_id in range(n_clusters):
    idxs = [i for i, label in enumerate(labels) if label == cluster_id]
    cluster_embeddings = [embeddings[i] for i in idxs]
    sims = cosine_similarity([kmeans.cluster_centers_[cluster_id]], cluster_embeddings)
    central_idx = idxs[np.argmax(sims)]
    representatives.append((cluster_id, questions[central_idx]))

In [12]:
pattern = r'^\s*\*{0,2}(\d+)\.?\*{0,2}\s'                                                       
print("Representative Questions from Each Cluster:\n")
cluster_map={}
for cluster_id, question in representatives:
    fixed_question = question.encode('unicode_escape').decode()
    match = re.search(pattern, fixed_question)
    if match:
        # Access the captured group using .group(1)
        extracted_substring = int(match.group(1).strip())
        cluster_map[extracted_substring] = fixed_question
    else:
        print("No match found.")
for id in sorted(cluster_map.keys()):
    print(cluster_map[id])

Representative Questions from Each Cluster:

No match found.
 4. Let for \\( i = 1, 2, 3, p_i(x) \\) be a polynomial of degree \\( 2 \\) in \\( x \\), \\( p_i'(x) \\) and \\( p_i''(x) \\) be the first and second order derivatives of \\( p_i(x) \\) respectively. Let, \\( A(x) = p_1(x)p_2(x)p_3(x) \\) and \\( B(x) = [A(x)]^T A(x) \\). Then:    (1) \\( \\text{determinant of } B(x) \\)      (2) is a polynomial of degree \\( 6 \\) in \\( x \\).      (3) is a polynomial of degree \\( 3 \\) in \\( x \\).      (4) does not depend on \\( x \\).  \n
 5. If \\( A = \\begin{pmatrix} 0 & -1 \\\\ 1 & 0 \\end{pmatrix} \\), then which one of the following statements is not correct?    (1) \\( A^{-1} = A(-A) \\)      (2) \\( A^3 = I \\)      (3) \\( A^2 = I \\)      (4) \\( A^{-1} = A^2 + I \\)  \n
 8. Suppose \\( A \\) is any \\( 3 \\times 3 \\) non-singular matrix and \\( (A - 5I) = O \\), where \\( I = I_3 \\) and \\( O = O_3 \\). If \\( \\alpha A + \\beta A^{-1} = 4 \\), then \\( \\alpha + \\beta \

In [81]:
with open('output/inv-trig/rep-questions.md', 'w') as rqf:
    for id in sorted(cluster_map.keys()):
        rqf.write(cluster_map[id] + '\n')

In [13]:
df_clusters = pd.DataFrame({
    "Question": questions,
    "Cluster": labels
})

In [14]:
pd.set_option("display.max_colwidth", 60)
print(df_clusters.sort_values("Cluster"))

                                                       Question  Cluster
22  23.    If \( f(x) = \frac{1 + \sin x \cos x}{\sin x} \) ...        0
28     29. If \( f(x) = \int_0^{x} \frac{f(x)}{g(\sqrt{2})}d...        0
39  40.  The number of distinct real solutions of the equati...        0
1    2. Let \( A \) and \( B \) be any \( 2 \times 2 \) matr...        1
45  46.  If \( A = \begin{pmatrix} -2 & -1 \\ 4 & 2 \end{pma...        1
36  37.  Matrix A such that \( A^2 = 2A - I \), where I is t...        1
23  24.    If \( A = \begin{pmatrix} C_{n}^{2} & n^{3}C_{2} ...        2
52  53.    Matrix \( A \) is given by \( A = \begin{bmatrix}...        2
53  54.    If the matrix \( A = \begin{bmatrix} a & b & c \\...        2
54  55.    If \( \lambda(x^2 + 4x - 2) = 2 = a x^3 + b x^2 +...        2
34     35. Let \[ A = \begin{bmatrix} 1 & 0 & 0 \\ 2 & x & 0...        2
27  28.    If \( AB = A \) and \( BA = B \), then:   (1) \( ...        2
35     36. If   \[ A = \begin{bmatrix}  a^2 + x^2 &

In [15]:
# Specify the cluster you want to print questions from
target_cluster = 3  # Replace with your desired cluster number

# Filter questions belonging to the target cluster
cluster_questions = df_clusters[df_clusters['Cluster'] == target_cluster]

print(f"Questions in Cluster {target_cluster}:\n")
for index, row in cluster_questions.iterrows():
    fixed_question = row['Question'].encode('unicode_escape').decode()
    print(fixed_question)

Questions in Cluster 3:

10.    If \\( A \\) is a \\( 3 \\times 3 \\) matrix such that \\( |5 \\cdot \\text{adj} A| = 5 \\), then \\( |A| \\) is equal to:   (1) \\( \\frac{1}{5} \\)   (2) \\( \\frac{1}{25} \\)   (3) \\( \\pm 1 \\)   (4) \\( \\pm 5 \\)  \n
11.    Let \\( P = \\begin{bmatrix} \\sqrt{3} & 1 & -\\sqrt{3} \\\\ 1 & 0 & 0 \\\\ 2 & 2 & 2 \\end{bmatrix} \\), \\( A = \\begin{bmatrix} 0 & -1 & 0 \\\\ 2 & 2 & 0 \\\\ 2015 & 0 & 0 \\end{bmatrix} \\) and \\( Q = P A P^T \\), then \\( P T_{Q}^{2015} P \\) is:   (1) \\( \\begin{bmatrix} 0 & 2015 \\\\ 1 & 0 \\end{bmatrix} \\)   (2) \\( \\begin{bmatrix} 0 & 0 \\\\ 2015 & 1 \\end{bmatrix} \\)   (3) \\( \\begin{bmatrix} 0 & 2015 \\\\ 0 & 0 \\end{bmatrix} \\)   (4) \\( \\begin{bmatrix} 2015 & 0 \\\\ 1 & 2015 \\end{bmatrix} \\)  \n
43.  Let \\( A = \\begin{pmatrix} 0 & 2 & q \\\\ p & -q & r \\end{pmatrix} \\). If \\( AAT = I_3 \\), then \\( |p| \\) is: (1) \\( \\frac{1}{5} \\)   (2) \\( \\frac{1}{3} \\)   (3) \\( \\frac{1}{6} \\)   (4) \\( \

In [65]:
approach_embeddings = model.encode(approaches)

In [66]:
with open('output/inv-trig/test/all-test-questions-approaches.md', 'r') as qf:
    test_approaches = qf.readlines()

In [67]:
# Encode test questions
test_approach_embeddings = model.encode(test_approaches)

closest_approaches_info = []

for i, test_approach_embedding in enumerate(test_approach_embeddings):
    similarities = cosine_similarity([test_approach_embedding], approach_embeddings)
    closest_idx = np.argmax(similarities)
    closest_approach = approaches[closest_idx]

    # Calculate similarity to closest question
    sim_to_closest_question = cosine_similarity(
        [test_approach_embedding], 
        [approach_embeddings[approaches.index(closest_approach)]]
    )[0][0]

    closest_approaches_info.append({
        "Test Approach": test_approaches[i],
        "Closest Approach": closest_approach,
        "Similarity to Closest Approach": sim_to_closest_question
    })

In [69]:
# Display the evaluation results
print("Test Approach | Closest Approach | Similarity")
for info in closest_approaches_info:
    print(f"{info['Test Approach'].strip()} | {info['Closest Approach']} | {info['Similarity to Closest Approach']}")

Test Approach | Closest Approach | Similarity
1: Set x = cos α and use cos 3α = 4cos^3α − 3cosα, determine α’s interval from x∈(−1,−1/2), reduce 3α by 2π to the principal arccos range [0,π], use arcsin x = π/2 − α, then add and simplify. | 30: Reduce 12 modulo 2π, identify which subintervals of the principal ranges arcsin∈[-π/2,π/2] and arccos∈[0,π] the reduced angle lies in, apply the piecewise principal-value formulas for arcsin(sin x) and arccos(cos x) (using sine/cosine symmetry and parity to choose the correct branch), then sum and simplify. | 0.7367293834686279
2: Use principal-value identity sec^{-1}x + csc^{-1}x = π/2, substitute to get a quadratic in one inverse-angle, simplify algebraically, find its vertex (complete the square or differentiate) for the minimum within the allowed principal-value interval and evaluate the interval endpoints for the maximum, then add those two values. | 1: Recall standard cosine values and their principal arccos angles, map each given value to 